In [2]:
import os
import re

folder = '../data/raw/traffic_volume'  # 실제 경로로 수정

for fname in os.listdir(folder):
    if not fname.endswith('.xlsx'):
        continue
    
    # "01월 서울시 교통량 조사자료(2025).xlsx" → month="01", year="2025"
    match = re.match(r'(\d{2})월\s.*\((\d{4})\)\.xlsx', fname)
    if match:
        month = match.group(1)
        year = match.group(2)
        new_name = f'{year}_{month}.xlsx'
        
        old_path = os.path.join(folder, fname)
        new_path = os.path.join(folder, new_name)
        
        print(f'{fname}  →  {new_name}')
        os.rename(old_path, new_path)

print('\n완료!')

01월 서울시 교통량 조사자료(2025).xlsx  →  2025_01.xlsx
02월 서울시 교통량 조사자료(2025).xlsx  →  2025_02.xlsx
03월 서울시 교통량 조사자료(2025).xlsx  →  2025_03.xlsx
04월 서울시 교통량 조사자료(2025).xlsx  →  2025_04.xlsx
05월 서울시 교통량 조사자료(2025).xlsx  →  2025_05.xlsx
06월 서울시 교통량 조사자료(2025).xlsx  →  2025_06.xlsx
07월 서울시 교통량 조사자료(2025).xlsx  →  2025_07.xlsx
08월 서울시 교통량 조사자료(2025).xlsx  →  2025_08.xlsx
09월 서울시 교통량 조사자료(2025).xlsx  →  2025_09.xlsx
10월 서울시 교통량 조사자료(2025).xlsx  →  2025_10.xlsx
11월 서울시 교통량 조사자료(2025).xlsx  →  2025_11.xlsx
12월 서울시 교통량 조사자료(2025).xlsx  →  2025_12.xlsx

완료!


In [1]:
# notebooks/01_data_check.ipynb

import sys
sys.path.insert(0, '..')

from utils import *
import pandas as pd
import geopandas as gpd

# === 1. 사고 데이터 ===
acc = pd.read_csv(ACCIDENTS)
print("=== 사고 데이터 ===")
print(f"행: {len(acc)}, 열: {len(acc.columns)}")
print(acc.columns.tolist())
print(acc[['lon','lat']].describe())
print(acc.head(3))
print()

# === 2. 횡단보도 ===
cw = pd.read_csv(CROSSWALK, encoding='cp949')
print("=== 횡단보도 ===")
print(f"행: {len(cw)}, 열: {len(cw.columns)}")
print(cw.columns.tolist())
print(cw.head(3))
print()

# === 3. 보행자신호등 ===
ped = pd.read_csv(PED_SIGNAL, encoding='cp949')
print("=== 보행자신호등 ===")
print(f"행: {len(ped)}, 열: {len(ped.columns)}")
print(ped.columns.tolist())
print(ped.head(3))
print()

# === 4. 고령인구 === 
eld = pd.read_csv(ELDERLY_POP_DONG, encoding='utf-8-sig', header=None, nrows=10)
print("=== 고령인구 (동별) ===")
print(eld.to_string())
print()

# 실제 데이터 (헤더 4줄 스킵)
eld = pd.read_csv(ELDERLY_POP_DONG, encoding='utf-8-sig', skiprows=4)
eld.columns = ['시도', '구', '동', '전체_소계', '전체_남', '전체_여',
               '노인_소계', '노인_남', '노인_여',
               '노인_내국_소계', '노인_내국_남', '노인_내국_여',
               '노인_외국_소계', '노인_외국_남', '노인_외국_여']
print(f"행: {len(eld)}, 열: {len(eld.columns)}")
print(eld.head(10))

# === 5. 표준노드링크 ===
link = gpd.read_file(MOCT_LINK_SHP, encoding='cp949')
print("=== 표준노드링크 (LINK) ===")
print(f"행: {len(link)}, 열: {len(link.columns)}")
print(link.columns.tolist())
print(link.head(3))
print()

# === 6. 교통량 (1월만 샘플) ===
tv = pd.read_excel(TRAFFIC_DIR / "2025_01.xlsx")
print("=== 교통량 ===")
print(f"행: {len(tv)}, 열: {len(tv.columns)}")
print(tv.columns.tolist())
print(tv.head(3))

=== 사고 데이터 ===
행: 4588, 열: 22
['구분번호', 'acdnt_no', 'acdnt_year', 'acdnt_dd_dc', 'tmzon', 'acdnt_dc', 'acdnt_gae_dc', 'dprs_cnt', 'sep_cnt', 'slp_cnt', 'lrg_violt', 'road_stle', 'wether', 'wrngdo_vhcle', 'dmge_age_2', 'injury_2', 'acdnt_pos', 'route_nm', 'x_crdnt', 'y_crdnt', 'lon', 'lat']
               lon          lat
count  4587.000000  4587.000000
mean    126.994232    37.562236
std       0.017238     0.004769
min     126.908953    37.514220
25%     126.978476    37.559005
50%     126.992564    37.562689
75%     127.010812    37.565848
max     127.025635    37.571669
         구분번호          acdnt_no  acdnt_year acdnt_dd_dc tmzon      acdnt_dc  \
0  2010000986  2010010200100450        2010   2010년  1월    야간     차대사람 - 기타   
1  2010001372  2010010300100328        2010   2010년  1월    야간     차대사람 - 기타   
2  2010001427  2010010300100383        2010   2010년  1월    야간  차대사람 - 보도통행중   

  acdnt_gae_dc  dprs_cnt  sep_cnt  slp_cnt  ... wether wrngdo_vhcle  \
0         경상사고         0        0 

In [13]:
# 사고 유형 확인 (이게 진짜 사고 유형 컬럼)
print("=== acdnt_dc 분포 ===")
print(acc['acdnt_dc'].value_counts())
print()

# 올바른 필터
elderly_cross = acc[
    (acc['dmge_age_2'] == '65세 이상') &
    (acc['acdnt_dc'].str.contains('횡단', na=False))
]
print(f"65세↑ + 횡단중: {len(elderly_cross)}건 / 전체 {len(acc)}건")
print()

# 65세 이상 전체 (횡단 아닌 것 포함)
elderly_all = acc[acc['dmge_age_2'] == '65세 이상']
print(f"65세↑ 전체: {len(elderly_all)}건")
print(elderly_all['acdnt_dc'].value_counts())

=== acdnt_dc 분포 ===
acdnt_dc
차대사람 - 기타            1856
차대사람 - 횡단중           1614
차대사람 - 차도통행중          520
차대사람 - 보도통행중          318
차대사람 - 길가장자리구역통행중     280
Name: count, dtype: int64

65세↑ + 횡단중: 342건 / 전체 4588건

65세↑ 전체: 904건
acdnt_dc
차대사람 - 기타            353
차대사람 - 횡단중           342
차대사람 - 차도통행중         109
차대사람 - 길가장자리구역통행중     56
차대사람 - 보도통행중          44
Name: count, dtype: int64


In [12]:
import pandas as pd
import os

# === 1. 지점 좌표 테이블 (시트 2, 아무 월이나 하나에서) ===
stations = pd.read_excel(
    TRAFFIC_DIR / "2025_01.xlsx", 
    sheet_name=2
)
print("=== 교통량 지점 ===")
print(f"지점 수: {len(stations)}")
print(stations.columns.tolist())
print(stations.head(3))
print()

# 중구 근처 지점 확인 (위도 37.55~37.57, 경도 126.97~127.01 정도)
stations_junggu = stations[
    (stations['위도'] > 37.54) & (stations['위도'] < 37.58) &
    (stations['경도'] > 126.96) & (stations['경도'] < 127.02)
]
print(f"중구 근처 지점: {len(stations_junggu)}개")
print(stations_junggu[['지점번호', '지점명칭', '위도', '경도']])
print()

# === 2. 12개월 교통량 합치기 ===
all_traffic = []
for fname in sorted(os.listdir(TRAFFIC_DIR)):
    if not fname.endswith('.xlsx'):
        continue
    df = pd.read_excel(TRAFFIC_DIR / fname, sheet_name=1)
    all_traffic.append(df)
    print(f"  {fname}: {len(df)}행")

traffic = pd.concat(all_traffic, ignore_index=True)
print(f"\n전체 교통량: {len(traffic)}행")
print(traffic.columns.tolist())
print()

# === 3. 지점별 일평균 교통량 (AADT) ===
# 시간대 컬럼 합산 → 일 교통량
hour_cols = [f'{h}시' for h in range(24)]
traffic['일교통량'] = traffic[hour_cols].apply(
    lambda row: pd.to_numeric(row, errors='coerce').sum(), axis=1
)

# 지점별 연평균
aadt = traffic.groupby('지점번호')['일교통량'].mean().reset_index()
aadt.columns = ['지점번호', 'AADT']
print("=== 지점별 AADT ===")
print(aadt.head(10))

=== 교통량 지점 ===
지점 수: 147
['지점번호', '지점명칭', '검지기 유형', '위도', '경도', '주소', '도로명 주소', '유입 방향', '유출방향']
   지점번호         지점명칭  검지기 유형         위도          경도                  주소  \
0  A-01    성산로(금화터널)     지자기  37.568588  126.948436  서울시 서대문구 신촌동 1-142   
1  A-02    사직로(사직터널)     지자기  37.572298  126.962853   서울시 종로구 행촌동 1-186   
2  A-03  자하문로(자하문터널)  영상(AI)  37.588831  126.968548    서울시 종로구 청운동 7-20   

  도로명 주소              유입 방향               유출방향  
0    NaN  [성산로]봉원고가차도->독립문역  [성산로]독립문역->봉원고가차도  
1    NaN     [사직로]독립문역->사직단     [사직로]사직단->독립문역  
2    NaN  [자하문로]석파정->청운초등학교  [자하문로]청운초등학교->석파정  

중구 근처 지점: 20개
    지점번호            지점명칭         위도          경도
1   A-02       사직로(사직터널)  37.572298  126.962853
4   A-05        율곡로(안국역)  37.576000  126.984342
6   A-07  대학로(한국방송통신대학교)  37.578188  127.002053
7   A-08        종로(동묘앞역)  37.573262  127.017112
9   A-10      동호로(장충체육관)  37.558563  127.007101
10  A-11     장충단로(장충단공원)  37.556889  127.004655
11  A-12        퇴계로(회현역)  37.557432  126.976486
12  A-1